# Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
- A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

c:\Users\sk335\OneDrive\Documents\Coding\PythonProjects\langchain-and-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by breaking down what I know about parrots and their ability to mimic human speech. First, parrots are part of the Psittaciformes order, which includes various species like macaws, cockatiels, and budgerigars. They\'re known for their vocal mimicry. But why do they do this?\n\nI remember reading that parrots have a unique vocal organ called the syrinx, which allows them to produce a wide range of sounds. Unlike humans, who use vocal cords, the syrinx is located at the base of the trachea and can generate complex sounds. That\'s probably why they can imitate human speech so well. But is that the only reason?\n\nAnother angle is their social behavior. Parrots are highly social animals. In the wild, they live in flocks and use calls to communicate with each other. I think vocal mimicry might be a form of social bonding. By mimicking sounds, they can integrate into the group and show they\'re part of the

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I'll structure the tool call with the location parameter set to Boston. Make sure the JSON is correctly formatted. No other parameters are needed here. Just call get_weather with location: Boston.\n", 'tool_calls': [{'id': '57177aeq1', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 154, 'total_tokens': 256, 'completion_time': 0.174451137, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.00693686, 'prompt_tokens_details': None, 'queue_time': 0.05251707, 'total_time': 0.181387997}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_cal

# Tool Execution Loops

In [4]:
# Step 1: Model generate tool calls
messages = [{"role": "user", "content": "What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# Step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Boston is sunny right now. It's a great day to enjoy outdoor activities!


In [5]:
messages

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': '95cg5ss0f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.138419235, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.007187926, 'prompt_tokens_details': None, 'queue_time': 0.051947865, 'total_time': 0.145607161}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_cal